# Thermal Detective — chip-temperature PINN
Runs every stage in order. Each cell calls a plain `.py` script, so all outputs land in
`/kaggle/working/chip-pinn/results/` and survive **Save Version**.

## Setup — get the code into /kaggle/working/chip-pinn

In [ ]:
import os, subprocess, sys
PROJECT = "/kaggle/working/chip-pinn" if os.path.isdir("/kaggle/working") else os.path.abspath(".")
REPO_URL = "https://github.com/cosmicwonder338-oss/ireProj2.git"   # repo that contains chip-pinn/
BRANCH = "claude/thermal-detective-pinn-8b23c9"   # branch that contains chip-pinn/ (main once merged)
if not os.path.isdir(os.path.join(PROJECT, "common")) and os.path.isdir("/kaggle/working"):
    # Option A: attach the code as a Kaggle Dataset and copy it:  !cp -r /kaggle/input/<dataset>/chip-pinn /kaggle/working/
    # Option B: clone from GitHub (needs Internet ON in notebook settings):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "/kaggle/working/_repo"], check=True)
    subprocess.run(["cp", "-r", "/kaggle/working/_repo/chip-pinn", "/kaggle/working/"], check=True)
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
print("project:", os.getcwd())

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'no GPU'

## Stage 0 — infrastructure smoke test + unit tests

In [ ]:
!python scripts/stage0_smoke.py --config configs/stage0_smoke.yaml

In [ ]:
!python tests/run_tests.py

## Stage 1 — finite-volume reference solutions (3 layouts × 2 cooling models) + sensors

In [ ]:
!python scripts/stage1_reference.py --config configs/stage1_reference.yaml

In [ ]:
import pandas as pd
from IPython.display import Image, display
df = pd.read_csv("results/stage1_reference/metrics.csv")
display(df[["layout","cooling","T_max_C","T_min_C","T_range_C","hotspot_to_q_peak_mm","P_total_W","q_peak_W_cm2","solve_time_s","energy_balance_rel_err"]].round(3))
for c in ("edge", "backside"):
    display(Image(f"results/stage1_reference/layouts_{c}.png"))

## Stage 2 — verification: FV convergence (manufactured solution) + autograd residuals

In [ ]:
!python tests/run_tests.py solver autograd

In [ ]:
!python scripts/stage2_verify.py --config configs/stage2_verify.yaml

In [ ]:
display(Image("results/stage2_verify/convergence.png"))